# SongFormer: Music Structure Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SidSaxena/SongFormer/blob/main/notebooks/SongFormer-Colab.ipynb)

**SongFormer** segments songs into labeled sections (intro, verse, chorus, bridge, …).

This notebook runs the [fork](https://github.com/SidSaxena/SongFormer) of SongFormer with its upgraded Gradio app:

- **Option A — Gradio Web App**: analyze a single file with downloadable results (JSON / MSA / CSV / plot / ZIP), or use the **Batch tab** — live per-file status, a combined ZIP you can download mid-run, and a per-file inspector with audio playback.
- **Option B — Headless batch inference (advanced)**: the command-line `.scp` pipeline, for large libraries or scripting.

**Requirements:** a GPU runtime is recommended (`Runtime → Change runtime type → GPU`).

The setup sections below are collapsed by default — `Runtime → Run all` executes them anyway; expand any section to inspect it.

Paper: [arXiv:2510.02797](https://arxiv.org/abs/2510.02797) | Upstream: [ASLP-lab/SongFormer](https://github.com/ASLP-lab/SongFormer) | Fork: [SidSaxena/SongFormer](https://github.com/SidSaxena/SongFormer) | [HF Space](https://huggingface.co/spaces/SidSaxena/SongFormer)


---
## 1. Check GPU

In [ ]:
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    mem = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f"GPU Memory: {mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime → Change runtime type → GPU")

---
## 2. Clone repository and install dependencies

Uses Colab's pre-installed PyTorch — no need to reinstall.

> **If installation or the import verification fails** (typically after Colab upgrades its pre-installed libraries): expand this section — it contains **Troubleshooting** notes and a **Fallback** cell that installs pinned, known-good versions (PyTorch 2.8.0 + dependencies). Tick `RUN_FALLBACK` there and re-run it.


In [ ]:
!git clone https://github.com/SidSaxena/SongFormer.git
%cd SongFormer
!git submodule update --init --recursive


In [ ]:
# Install only packages Colab doesn't have (skip torch, numpy, scipy, etc.)
!pip install -q -r requirements-colab.txt

### Troubleshooting

If the verification cell below reports missing packages:
1. Restart the runtime: `Runtime → Restart session`
2. Re-run from the install cell above

If it still fails, run the **Fallback** cell at the bottom of this section to install PyTorch 2.8.0.

In [ ]:
# Verify all critical imports
# Monkey-patch for msaf (scipy.inf removed in newer scipy)
import scipy, numpy as np
scipy.inf = np.inf

import importlib
import sys

# Map: import_name -> display_name
required = {
    'torch': 'torch', 'torchaudio': 'torchaudio', 'ema_pytorch': 'ema_pytorch',
    'omegaconf': 'omegaconf', 'librosa': 'librosa', 'gradio': 'gradio',
    'transformers': 'transformers', 'safetensors': 'safetensors',
    'einops': 'einops', 'x_transformers': 'x_transformers',
    'muq': 'muq', 'scipy': 'scipy', 'numpy': 'numpy',
    'matplotlib': 'matplotlib', 'lightning': 'lightning', 'msaf': 'msaf',
    'hydra': 'hydra_core', 'PIL': 'pillow', 'sklearn': 'scikit_learn',
}

missing = []
for import_name, display_name in required.items():
    try:
        importlib.import_module(import_name)
    except ImportError:
        missing.append(display_name)

if missing:
    print(f"ERROR: Missing packages: {', '.join(missing)}")
    print("\nFix: Restart runtime (Runtime → Restart session) and re-run from the install cell.")
    print("If it still fails, run the Fallback cell below.")
    sys.exit(1)
else:
    print("All critical packages installed successfully!")
    print(f"  torch {torch.__version__}, gradio {importlib.import_module('gradio').__version__}")

### Fallback: Install PyTorch 2.8.0

**Only run this if the verification cell above reported errors.**

After running this cell:
1. Restart runtime: `Runtime → Restart session`
2. Re-run from the clone cell (step 2)

In [ ]:
# Only run if verification failed above
RUN_FALLBACK = False  #@param {type:"boolean"}

if RUN_FALLBACK:
    !pip install -q torch==2.8.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu128
    !pip install -q -r requirements-colab.txt
else:
    print("Skipped. Only run this if verification failed above.")

---
## 3. Download model checkpoints

Models total **~2.7 GB** per session:
- `pretrained_msd.pt` (MusicFM) — 1.23 GB
- `SongFormer.safetensors` — 99.6 MB
- MuQ encoder (HF cache) — 1.33 GB
- `msd_stats.json` — 2.2 KB

> **Optional — persist models on Google Drive:** saves re-downloading ~2.7 GB every session, at the cost of the same Drive space. Note: Drive reads can be *slower* than re-downloading from Hugging Face — this saves bandwidth and avoids download flakiness, not necessarily startup time. The toggle lives inside this collapsed section: expand it and tick **`STORE_ON_DRIVE`** *before* running the cell (with `Run all` it defaults to off).


In [ ]:
#@markdown ### Checkpoint Storage
#@markdown Persist checkpoints on Google Drive to avoid re-downloading each session?
STORE_ON_DRIVE = False  #@param {type:"boolean"}

import os
import shutil

REPO_CKPTS = '/content/SongFormer/src/SongFormer/ckpts'

if STORE_ON_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_CKPTS = '/content/drive/MyDrive/SongFormer/ckpts'
    os.makedirs(DRIVE_CKPTS, exist_ok=True)

    # Symlink Drive checkpoints into the repo
    if os.path.isdir(REPO_CKPTS) and not os.path.islink(REPO_CKPTS):
        for f in os.listdir(REPO_CKPTS):
            src = os.path.join(REPO_CKPTS, f)
            dst = os.path.join(DRIVE_CKPTS, f)
            if not os.path.exists(dst):
                if os.path.isdir(src):
                    shutil.copytree(src, dst)
                else:
                    shutil.copy2(src, dst)
        shutil.rmtree(REPO_CKPTS)
    elif not os.path.exists(REPO_CKPTS):
        pass

    if not os.path.islink(REPO_CKPTS):
        os.symlink(DRIVE_CKPTS, REPO_CKPTS)
    print(f"Checkpoints will be stored at: {DRIVE_CKPTS}")

    # Persist the MuQ encoder cache (HF hub cache) on Drive too, so the
    # ~1.33 GB MuQ download survives across sessions. Symlink the cache
    # dir (env vars are resolved at import time, too late here).
    HF_CACHE = '/root/.cache/huggingface'
    DRIVE_HF = '/content/drive/MyDrive/SongFormer/hf_cache'
    os.makedirs(DRIVE_HF, exist_ok=True)
    if not os.path.islink(HF_CACHE):
        if os.path.isdir(HF_CACHE):
            for item in os.listdir(HF_CACHE):
                dst = os.path.join(DRIVE_HF, item)
                if not os.path.exists(dst):
                    shutil.move(os.path.join(HF_CACHE, item), dst)
            shutil.rmtree(HF_CACHE)
        os.makedirs(os.path.dirname(HF_CACHE), exist_ok=True)
        os.symlink(DRIVE_HF, HF_CACHE)
    print(f"HF model cache (MuQ) stored at: {DRIVE_HF}")
else:
    print("Checkpoints will be downloaded to the session (not persisted).")

# Download any missing checkpoints
os.chdir('/content/SongFormer/src/SongFormer')
from utils.fetch_pretrained import download_all
download_all()
os.chdir('/content/SongFormer')

# Pre-download MuQ model to HF cache (~1.33 GB)
print("\nPre-downloading MuQ model...")
from muq import MuQ
MuQ.from_pretrained("OpenMuQ/MuQ-large-msd-iter")
print("MuQ model cached.")

# Verify
print("\nDownloaded checkpoints:")
for f in ['MusicFM/msd_stats.json', 'MusicFM/pretrained_msd.pt', 'SongFormer.safetensors']:
    path = os.path.join(REPO_CKPTS, f)
    size_mb = os.path.getsize(path) / 1e6
    print(f"  {f} ({size_mb:.1f} MB)")

---
## Option A: Gradio Web App

Run the cell below, then click the public URL that appears (e.g. `https://xxxxx.gradio.live`).

**Single File tab** — upload a track from your computer, click **Analyze Music Structure**, and download the results as JSON, MSA text, CSV, plot PNG — or everything at once as a ZIP.

**Batch tab** — upload several files and analyze them sequentially: live per-file status (queued → processing → done), a combined ZIP with per-song folders plus `summary.csv` / `combined.json` manifests (downloadable mid-run), and a per-file inspector with segments table, JSON/MSA text, activation plot, and audio playback.


> **Note:** this cell keeps running while the app is live (that's normal). To continue with Option B below, stop the cell first (■ in the toolbar). `Runtime → Run all` therefore stops here by design.


In [ ]:
#@title 🚀 Launch the Gradio app (stop this cell to continue to Option B)
%cd /content/SongFormer
!SONGFORMER_SHARE=1 python app.py

---
## Option B: Headless Batch Inference (advanced)

The command-line `.scp` pipeline — useful for large libraries, scripting, or running without the UI. For a handful of files, the Gradio **Batch tab** in Option A is easier.

**How to use:**
1. Load audio from a Drive folder (B1) or upload manually (B2)
2. Create the `.scp` file and run inference (B3)
3. Download results (B4)


### B1. Audio Source: Google Drive (optional)

Load audio for batch inference. Choose one:
- **Google Drive folder** — enter a path below, files are copied to `/content/audio` automatically
- **Upload manually** — leave empty and use B2 instead

Supports full paths (`/content/drive/MyDrive/my_songs`) or relative to Drive root (`my_songs`).

Audio formats: `.mp3`, `.wav`, `.flac`, `.ogg`, `.m4a`, `.aac`, `.wma`


In [ ]:
#@markdown ### Audio Source
#@markdown Enter a Google Drive folder path, or leave empty to upload manually.
DRIVE_AUDIO_FOLDER = ""  #@param {type:"string"}

import os
import shutil

AUDIO_DIR = '/content/audio'
os.makedirs(AUDIO_DIR, exist_ok=True)
AUDIO_EXTENSIONS = ('.mp3', '.wav', '.flac', '.ogg', '.m4a', '.aac', '.wma')

drive_audio_available = False

if DRIVE_AUDIO_FOLDER.strip():
    # Mount Drive if not already mounted
    if not os.path.exists('/content/drive/MyDrive'):
        from google.colab import drive
        drive.mount('/content/drive')

    # Resolve path (support both full and relative paths)
    src = DRIVE_AUDIO_FOLDER.strip()
    if not src.startswith('/content/'):
        src = f'/content/drive/MyDrive/{src}'

    if os.path.isdir(src):
        count = 0
        for f in sorted(os.listdir(src)):
            if f.lower().endswith(AUDIO_EXTENSIONS):
                shutil.copy2(os.path.join(src, f), AUDIO_DIR)
                count += 1
        if count > 0:
            print(f"Copied {count} audio files from: {src}")
            drive_audio_available = True
        else:
            print(f"No audio files found in: {src}")
            print(f"Supported formats: {', '.join(AUDIO_EXTENSIONS)}")
    else:
        print(f"Directory not found: {src}")
        print("Please check the path and run this cell again.")
else:
    print("No Drive folder specified.")
    print("Upload audio files in the next cell (B2).")

# Show current audio files
audio_files = sorted([
    f for f in os.listdir(AUDIO_DIR)
    if f.lower().endswith(AUDIO_EXTENSIONS)
]) if os.path.isdir(AUDIO_DIR) else []

if audio_files:
    print(f"\nAudio files in {AUDIO_DIR} ({len(audio_files)} total):")
    for f in audio_files:
        size_mb = os.path.getsize(os.path.join(AUDIO_DIR, f)) / 1e6
        print(f"  {f} ({size_mb:.1f} MB)")

### B2. Upload audio files

Skip this cell if you already loaded files from a Drive folder in B1.


In [ ]:
#@markdown ### Upload Audio Files
#@markdown Skip if you already specified a Drive folder above.

import os

AUDIO_DIR = '/content/audio'
AUDIO_EXTENSIONS = ('.mp3', '.wav', '.flac', '.ogg', '.m4a', '.aac', '.wma')

# Check if audio files already exist (from Drive)
existing_files = sorted([
    f for f in os.listdir(AUDIO_DIR)
    if f.lower().endswith(AUDIO_EXTENSIONS)
]) if os.path.isdir(AUDIO_DIR) else []

if existing_files:
    print(f"Already have {len(existing_files)} audio files from Drive:")
    for f in existing_files:
        print(f"  {f}")
    print("\nUpload more files below, or skip this cell.")

# Upload widget
from google.colab import files
print("\nUpload audio files (mp3, wav, flac, etc.):")
uploaded = files.upload()
for filename, data in uploaded.items():
    dst = os.path.join(AUDIO_DIR, filename)
    with open(dst, 'wb') as f:
        f.write(data)
    print(f"  Saved: {dst}")

# Show all files
all_files = sorted([
    f for f in os.listdir(AUDIO_DIR)
    if f.lower().endswith(AUDIO_EXTENSIONS)
])
print(f"\nTotal audio files: {len(all_files)}")

### B3. Create `.scp` file and run inference

In [ ]:
#@title 📝 Create the .scp file list
import os

# Create .scp file (one audio path per line)
AUDIO_DIR = '/content/audio'
SCP_PATH = '/content/audio_list.scp'
OUTPUT_DIR = '/content/SongFormer/output'
AUDIO_EXTENSIONS = ('.mp3', '.wav', '.flac', '.ogg', '.m4a', '.aac', '.wma')

audio_files = sorted([
    os.path.join(AUDIO_DIR, f)
    for f in os.listdir(AUDIO_DIR)
    if f.lower().endswith(AUDIO_EXTENSIONS)
])

if not audio_files:
    print("ERROR: No audio files found!")
    print("Load a Drive folder in B1 or upload files in B2.")
else:
    with open(SCP_PATH, 'w') as f:
        for path in audio_files:
            f.write(path + '\n')

    print(f"Created {SCP_PATH} with {len(audio_files)} files")
    print("Files:")
    for p in audio_files:
        print(f"  {os.path.basename(p)}")

In [ ]:
#@title ▶️ Run batch inference
%cd /content/SongFormer/src/SongFormer

!PYTHONPATH=.:../third_party:$PYTHONPATH python infer/infer.py \
    --input_path /content/audio_list.scp \
    --output_path /content/SongFormer/output \
    --model SongFormer \
    --checkpoint SongFormer.safetensors \
    --config_path SongFormer.yaml \
    --gpu_num 1 \
    --save_plots

print("\n=== Results ===")
print("(activation plots + heatmaps are in output/plots/, included in the B4 ZIP)")
import json
if os.path.isdir('/content/SongFormer/output'):
    for f in sorted(os.listdir('/content/SongFormer/output')):
        if f.endswith('.json'):
            print(f"\n{f}:")
            with open(f'/content/SongFormer/output/{f}') as fh:
                data = json.load(fh)
                for seg in data:
                    print(f"  {seg['start']:.1f}s - {seg['end']:.1f}s: {seg['label']}")
else:
    print("No output directory. Check for errors above.")

### B4. Download results

The cell below zips the results for browser download, and can optionally also save the ZIP to your Drive (`SAVE_TO_DRIVE` toggle — timestamped, never overwrites).


In [ ]:
#@title ⬇️ Zip and download results
SAVE_TO_DRIVE = False  #@param {type:"boolean"}
#@markdown Tick to also copy the ZIP to `MyDrive/SongFormer/results/` (timestamped).
import os
import shutil
from google.colab import files

OUTPUT_DIR = '/content/SongFormer/output'
ZIP_PATH = '/content/songformer_results.zip'

if os.path.isdir(OUTPUT_DIR) and os.listdir(OUTPUT_DIR):
    shutil.make_archive(ZIP_PATH.replace('.zip', ''), 'zip', OUTPUT_DIR)
    print(f"Zipped results to: {ZIP_PATH}")
    if SAVE_TO_DRIVE:
        try:
            if not os.path.exists('/content/drive/MyDrive'):
                from google.colab import drive
                drive.mount('/content/drive')
            from datetime import datetime
            results_dir = '/content/drive/MyDrive/SongFormer/results'
            os.makedirs(results_dir, exist_ok=True)
            stamp = datetime.now().strftime('%Y%m%d-%H%M%S')
            drive_zip = os.path.join(results_dir, f'songformer_results_{stamp}.zip')
            shutil.copy2(ZIP_PATH, drive_zip)
            print(f"Saved to Drive: {drive_zip}")
        except Exception as e:
            print(f"Drive save failed ({e}); continuing with browser download.")
    files.download(ZIP_PATH)
else:
    print("No results to download. Run inference first (B3).")